# 05 — BERTopic for wondr

Topic modeling pipeline untuk wondr by BNI menggunakan BERTopic dengan IndoBERT embeddings.

**Pipeline overview:**
1. Load preprocessed data + generate (or load cached) IndoBERT embeddings
2. Sensitivity analysis 3×3 grid (UMAP n_neighbors × HDBSCAN min_cluster_size)
3. Pick winner via c_v coherence (filter by topic count 15-30)
4. Validate winner with c_npmi
5. Final BERTopic fit with winner params
6. Inspect topics, refine domain stopwords (Phase B)
7. Re-fit with extended stopwords
8. DTM (topics_over_time) — monthly granularity
9. Cosine similarity matrices per topic (semantic drift)

**Methodology reference:** `bertopic_session_handoff_v2.md`

## Section 0 — Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

from utils.embedding import load_indobert_model, generate_embeddings

## Section 1 — Load data + Generate embeddings

**Heads up:** First run akan generate embeddings dari scratch (~20-30 menit untuk wondr di CPU). Subsequent runs akan load dari cache (`data/embeddings/wondr_embeddings.npy`) — instant.

### 1.1 Load preprocessed data

In [4]:
df_wondr = pd.read_csv('data/processed/wondr_bertopic.csv')

print(f"Shape: {df_wondr.shape}")
print(f"Columns: {df_wondr.columns.tolist()}")
df_wondr.head()

Shape: (8982, 6)
Columns: ['review_id', 'review_text_cleaned', 'relative_month', 'relative_week', 'date_wib', 'rating']


,review_id,review_text_cleaned,relative_month,relative_week,date_wib,rating
0,05c4afd9-2b73-462f-93ed-ad2f853647b5,kok tidak bisa di screenshot bagaimana caranya...,1,1,2024-07-05 16:24:39,2
1,e4786af6-59c9-4f34-897f-d20f8c5f412b,mau daftar susah disuruh telpon bni tapi tidak...,1,1,2024-07-05 16:28:04,1
2,cc78e674-330f-46f1-b279-27a9cbe2d823,tidak bisa buka rekening padahal no hp sudah s...,1,1,2024-07-05 21:14:52,1
3,2441c0dc-eda3-4f2c-8abe-f860c951af50,fitur paling basic untuk cetak statement bulan...,1,1,2024-07-05 22:04:41,1
4,0afc898b-3b64-426d-acaf-ae6b3c4bc967,sangat bagus sekali tapi tidak ada pitur setor...,1,1,2024-07-05 23:54:54,2


In [5]:
# Sanity check: no nulls in critical columns
critical_cols = ['review_text_cleaned', 'relative_month', 'relative_week']
print("Null counts:")
print(df_wondr[critical_cols].isnull().sum())

Null counts:
review_text_cleaned    0
relative_month         0
relative_week          0
dtype: int64


In [6]:
# Sanity check: relative_month coverage (should be 1-12)
print("Relative month distribution:")
print(df_wondr['relative_month'].value_counts().sort_index())

Relative month distribution:
relative_month
1      302
2      395
3      544
4      678
5     2568
6     1297
7      727
8      621
9      462
10     357
11     388
12     643
Name: count, dtype: int64


### 1.2 Extract texts as list

BERTopic expects a list of strings, not a pandas Series. We extract once here and reuse throughout the notebook.

In [7]:
docs_wondr = df_wondr['review_text_cleaned'].tolist()

print(f"Total docs: {len(docs_wondr)}")
print(f"Sample doc: {docs_wondr[0][:200]}")

Total docs: 8982
Sample doc: kok tidak bisa di screenshot bagaimana caranya kasih bukti ke penerima bagaimana sih ini aplikasi


### 1.3 Load IndoBERT model

Loaded once and reused for all embedding operations. First call may take a few seconds (loading from local HuggingFace cache).

In [9]:
model = load_indobert_model()
print(f"Model loaded: {type(model).__name__}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: SentenceTransformer


### 1.4 Generate embeddings (or load from cache)

**First run:** generate from scratch, save to `data/embeddings/wondr_embeddings.npy` (~20-30 min on CPU).

**Subsequent runs:** load from cache (instant).

If preprocessing data changes, manually delete the cache file before re-running this cell.

In [10]:
embedding_cache_path = Path('data/embeddings/wondr_embeddings.npy')
embedding_cache_path.parent.mkdir(parents=True, exist_ok=True)

embeddings_wondr = generate_embeddings(
    texts=docs_wondr,
    model=model,
    cache_path=embedding_cache_path,
)

print(f"\nEmbedding shape: {embeddings_wondr.shape}")
print(f"Dtype: {embeddings_wondr.dtype}")
print(f"Cache file size: {embedding_cache_path.stat().st_size / (1024**2):.1f} MB")

[cache miss] Generating embeddings for 8982 texts...


Batches:   0%|          | 0/281 [00:00<?, ?it/s]

[cache miss] Saved to data\embeddings\wondr_embeddings.npy, shape: (8982, 768)

Embedding shape: (8982, 768)
Dtype: float32
Cache file size: 26.3 MB


## Section 2 — Sensitivity Analysis (3×3 Grid)

Run BERTopic with all 9 combinations of (n_neighbors, min_cluster_size) and record:
- Number of topics produced (excluding noise topic -1)
- c_v coherence score

Embedding is cached — only UMAP + HDBSCAN + c-TF-IDF re-run per combination. Estimated time: ~20-30 min total.

**Phase A stopwords:** Sastrawi only (~123 words). Domain-specific stopwords added in Phase B (Section 6-7).

In [11]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

from utils.clustering import build_umap_model, build_hdbscan_model
from utils.evaluation import compute_coherence
from utils.stopwords import get_sastrawi_stopwords

### 2.1 Setup grid + vectorizer config

The CountVectorizer config is shared across all 9 grid runs (only UMAP/HDBSCAN params vary).

In [14]:
# Grid search space
n_neighbors_grid = [10, 15, 25]
min_cluster_size_grid = [30, 60, 100]

# c-TF-IDF vectorizer (Phase A: Sastrawi stopwords only)
sastrawi_stopwords = get_sastrawi_stopwords()

vectorizer_model = CountVectorizer(
    stop_words=sastrawi_stopwords,
    ngram_range=(1, 2),
    min_df=2,        # was 10 — adjusted for topic-level doc count
    max_df=0.95,
)

print(f"Grid combinations: {len(n_neighbors_grid) * len(min_cluster_size_grid)}")
print(f"Sastrawi stopwords: {len(sastrawi_stopwords)} words")

Grid combinations: 9
Sastrawi stopwords: 123 words


### 2.2 Run grid search

For each combination: build UMAP + HDBSCAN, fit BERTopic with cached embeddings, compute c_v coherence, record results.

In [15]:
results = []

for nn in n_neighbors_grid:
    for mcs in min_cluster_size_grid:
        print(f"\n--- Fitting: n_neighbors={nn}, min_cluster_size={mcs} ---")
        
        umap_model = build_umap_model(n_neighbors=nn)
        hdbscan_model = build_hdbscan_model(min_cluster_size=mcs)
        
        topic_model = BERTopic(
            embedding_model=model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            top_n_words=10,
            verbose=False,
        )
        
        topics, _ = topic_model.fit_transform(docs_wondr, embeddings=embeddings_wondr)
        
        n_topics = len(set(topics)) - (1 if -1 in topics else 0)
        c_v = compute_coherence(topic_model, docs_wondr, coherence='c_v')
        
        print(f"  -> n_topics: {n_topics}, c_v: {c_v:.4f}")
        
        results.append({
            'n_neighbors': nn,
            'min_cluster_size': mcs,
            'n_topics': n_topics,
            'c_v': c_v,
        })

results_df = pd.DataFrame(results)
print("\n=== Grid Search Results ===")
print(results_df.to_string(index=False))


--- Fitting: n_neighbors=10, min_cluster_size=30 ---
  -> n_topics: 35, c_v: 0.6637

--- Fitting: n_neighbors=10, min_cluster_size=60 ---
  -> n_topics: 5, c_v: 0.6320

--- Fitting: n_neighbors=10, min_cluster_size=100 ---
  -> n_topics: 8, c_v: 0.6801

--- Fitting: n_neighbors=15, min_cluster_size=30 ---
  -> n_topics: 27, c_v: 0.6755

--- Fitting: n_neighbors=15, min_cluster_size=60 ---
  -> n_topics: 15, c_v: 0.7055

--- Fitting: n_neighbors=15, min_cluster_size=100 ---
  -> n_topics: 8, c_v: 0.6876

--- Fitting: n_neighbors=25, min_cluster_size=30 ---
  -> n_topics: 24, c_v: 0.6520

--- Fitting: n_neighbors=25, min_cluster_size=60 ---
  -> n_topics: 14, c_v: 0.6982

--- Fitting: n_neighbors=25, min_cluster_size=100 ---
  -> n_topics: 8, c_v: 0.6866

=== Grid Search Results ===
 n_neighbors  min_cluster_size  n_topics      c_v
          10                30        35 0.663652
          10                60         5 0.632037
          10               100         8 0.680051
       

## Section 3 — Pick Winner via c_v

Filter grid results by topic count target (15-30), then select highest c_v.

**Result of grid search:**
- 3 combinations passed the topic count filter
- Winner: `n_neighbors=15, min_cluster_size=60` with c_v=0.7055 (15 topics)

In [16]:
# Apply topic count filter (15-30 topics target per methodology)
filtered = results_df[(results_df['n_topics'] >= 15) & (results_df['n_topics'] <= 30)]

print("Combinations passing filter (15 <= n_topics <= 30):")
print(filtered.to_string(index=False))

# Pick winner: highest c_v from filtered
winner = filtered.loc[filtered['c_v'].idxmax()]

print(f"\n=== WINNER ===")
print(f"n_neighbors:      {int(winner['n_neighbors'])}")
print(f"min_cluster_size: {int(winner['min_cluster_size'])}")
print(f"n_topics:         {int(winner['n_topics'])}")
print(f"c_v:              {winner['c_v']:.4f}")

# Store as variables for later use
best_n_neighbors = int(winner['n_neighbors'])
best_min_cluster_size = int(winner['min_cluster_size'])

Combinations passing filter (15 <= n_topics <= 30):
 n_neighbors  min_cluster_size  n_topics      c_v
          15                30        27 0.675528
          15                60        15 0.705541
          25                30        24 0.651993

=== WINNER ===
n_neighbors:      15
min_cluster_size: 60
n_topics:         15
c_v:              0.7055


## Section 4 — Validate Winner with c_npmi

Re-fit BERTopic with winner params, compute both c_v and c_npmi for comparison.

**Why this matters:** c_v is our primary metric for model selection (per Röder et al., 2015). c_npmi serves as secondary validation — if both metrics agree the winner is good, we have stronger confidence. If they disagree, we discuss it as an interesting finding (still prefer c_v winner).

**Heads up:** Re-fit takes ~30-60 seconds (single fit, not full grid).

In [17]:
from utils.evaluation import compute_coherence_both

# Re-fit with winner params
print(f"Re-fitting BERTopic with n_neighbors={best_n_neighbors}, "
      f"min_cluster_size={best_min_cluster_size}...")

umap_winner = build_umap_model(n_neighbors=best_n_neighbors)
hdbscan_winner = build_hdbscan_model(min_cluster_size=best_min_cluster_size)

topic_model_winner = BERTopic(
    embedding_model=model,
    umap_model=umap_winner,
    hdbscan_model=hdbscan_winner,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=False,
)

topics_winner, _ = topic_model_winner.fit_transform(
    docs_wondr, embeddings=embeddings_wondr
)

# Compute both coherence scores
scores = compute_coherence_both(topic_model_winner, docs_wondr)

n_topics_winner = len(set(topics_winner)) - (1 if -1 in topics_winner else 0)

print(f"\n=== Validation Results ===")
print(f"n_topics: {n_topics_winner}")
print(f"c_v:      {scores['c_v']:.4f}  (primary metric)")
print(f"c_npmi:   {scores['c_npmi']:.4f}  (validation metric)")

Re-fitting BERTopic with n_neighbors=15, min_cluster_size=60...

=== Validation Results ===
n_topics: 15
c_v:      0.7055  (primary metric)
c_npmi:   0.1723  (validation metric)


### Interpretation

| Metric | Score | Notes |
|---|---|---|
| c_v | 0.7055 | Primary metric, used for model selection |
| c_npmi | 0.1723 | Secondary validation metric |

**Verdict:** Both metrics agree the winner combination is coherent. c_v at 0.7055 indicates good topic quality (range 0.7+ is "good"), and c_npmi at 0.1723 confirms positive co-occurrence correlation among top-10 words per topic (range 0.15-0.25 is "good" for c_npmi). No disagreement between primary and secondary metrics — proceed to final fit and Phase B refinement.

Winner params:
- `n_neighbors`: 15 (UMAP)
- `min_cluster_size`: 60 (HDBSCAN)
- Yields 15 topics

## Section 5: Final Phase A Model

Berdasarkan grid search di Section 2 dan validasi metrik di Section 3-4, parameter winner adalah **`n_neighbors=15`** (UMAP) dan **`min_cluster_size=60`** (HDBSCAN), menghasilkan **15 topik** dengan c_v=0.7055 dan c_npmi=0.1723. Kedua metrik koherensi sepakat (no disagreement), sehingga model ini di-lock sebagai **Phase A final model**.

Model `topic_model_winner` yang sudah di-fit di Section 4 digunakan langsung sebagai Phase A final — tidak perlu re-fit karena UMAP menggunakan `random_state=42` (deterministic). Cell berikutnya men-dokumentasikan konfigurasi final untuk reproducibility.

In [18]:
# ============================================================
# PHASE A FINAL MODEL — Configuration Summary
# ============================================================
# This cell documents the locked configuration for Phase A.
# Model `topic_model_winner` is already fitted (from Section 4).
# No re-fit needed: UMAP uses random_state=42 (deterministic).
# ============================================================

phase_a_config = {
    "embedding_model": "indobenchmark/indobert-base-p2",
    "embedding_dim": 768,
    "umap": {
        "n_neighbors": best_n_neighbors,          # 15
        "n_components": 5,
        "min_dist": 0.0,
        "metric": "cosine",
        "random_state": 42,
    },
    "hdbscan": {
        "min_cluster_size": best_min_cluster_size,  # 60
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "prediction_data": True,
    },
    "vectorizer": {
        "stop_words": "sastrawi (123 words)",
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
    },
    "results": {
        "n_topics": len(topic_model_winner.get_topic_info()) - 1,  # exclude outlier topic -1
        "c_v": 0.7055,
        "c_npmi": 0.1723,
    },
}

# Display nicely
import json
print("=" * 60)
print("PHASE A FINAL MODEL — LOCKED CONFIGURATION")
print("=" * 60)
print(json.dumps(phase_a_config, indent=2, default=str))
print("=" * 60)
print(f"\n✓ Model `topic_model_winner` ready for Section 6 (inspection)")

PHASE A FINAL MODEL — LOCKED CONFIGURATION
{
  "embedding_model": "indobenchmark/indobert-base-p2",
  "embedding_dim": 768,
  "umap": {
    "n_neighbors": 15,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
    "random_state": 42
  },
  "hdbscan": {
    "min_cluster_size": 60,
    "metric": "euclidean",
    "cluster_selection_method": "eom",
    "prediction_data": true
  },
  "vectorizer": {
    "stop_words": "sastrawi (123 words)",
    "ngram_range": [
      1,
      2
    ],
    "min_df": 2,
    "max_df": 0.95
  },
  "results": {
    "n_topics": 15,
    "c_v": 0.7055,
    "c_npmi": 0.1723
  }
}

✓ Model `topic_model_winner` ready for Section 6 (inspection)


## Section 6: Topic Inspection (Phase A Output)

Phase A menggunakan stopword umum Bahasa Indonesia (Sastrawi). Untuk meningkatkan diskriminabilitas antar topik, dilakukan inspeksi manual terhadap output Phase A guna mengidentifikasi **domain-specific stopwords** — yaitu kata yang secara teknis informatif (misal "aplikasi", "bank") tetapi muncul terlalu sering di banyak topik sehingga tidak diskriminatif.

Hasil inspeksi akan ditambahkan ke `CUSTOM_DOMAIN_STOPWORDS` di `utils/stopwords.py`, lalu BERTopic di-fit ulang di Section 7 (Phase B).

In [19]:
# ============================================================
# Section 6.1 — Topic Overview
# ============================================================
topic_info = topic_model_winner.get_topic_info()
print(f"Total topics (incl. -1 outlier): {len(topic_info)}")
print(f"Real topics: {len(topic_info) - 1}")
print(f"Outlier docs (topic -1): {topic_info.loc[topic_info['Topic'] == -1, 'Count'].values[0]}")
print()
print(topic_info[['Topic', 'Count', 'Name']].to_string(index=False))

Total topics (incl. -1 outlier): 16
Real topics: 15
Outlier docs (topic -1): 5958

 Topic  Count                                       Name
    -1   5958            -1_pakai_transaksi_malah_mobile
     0    450  0_wajah_verifikasi wajah_verifikasi_gagal
     1    416        1_saldo_gagal_gagal saldo_transaksi
     2    399 2_gangguan_sering_gangguan terus_perbaikan
     3    365          3_mobile_lebih_bni mobile_banking
     4    264            4_tunai_tarik_tarik tunai_kartu
     5    171 5_mobile_bni mobile_banking_mobile banking
     6    163                  6_kode_email_kode otp_otp
     7    152             7_kendala_buka_perbaiki_dibuka
     8    118           8_pasword_salah_verifikasi_wajah
     9    106              9_buka_login_kok_aplikasi nya
    10    103                10_login_sesi_daftar_syarat
    11     90                  11_jangan_siap_kalau_dulu
    12     87                    12_buka_kok_dibuka_hari
    13     72        13_perbaiki_diperbaiki_sering_mohon
    1

In [20]:
# ============================================================
# Section 6.2 — Top-10 words per topic (compact view)
# ============================================================
print(f"{'Topic':<6} | {'Top 10 words (with c-TF-IDF scores)'}")
print("=" * 90)
for topic_id in range(len(topic_info) - 1):  # skip -1 outlier
    words_scores = topic_model_winner.get_topic(topic_id)
    words_str = ", ".join([f"{w}({s:.2f})" for w, s in words_scores])
    print(f"T{topic_id:<5} | {words_str}")

Topic  | Top 10 words (with c-TF-IDF scores)
T0     | wajah(0.15), verifikasi wajah(0.13), verifikasi(0.12), gagal(0.07), gagal terus(0.06), wajah gagal(0.04), wajah susah(0.03), susah banget(0.03), padahal(0.03), mengikuti(0.03)
T1     | saldo(0.11), gagal(0.06), gagal saldo(0.06), transaksi(0.05), transaksi gagal(0.04), berkurang(0.04), terpotong(0.04), pulsa(0.04), kepotong(0.03), transfer(0.03)
T2     | gangguan(0.11), sering(0.10), gangguan terus(0.05), perbaikan(0.05), sering error(0.05), banget(0.04), maintenance(0.04), sering gangguan(0.04), jelek(0.03), seharian(0.03)
T3     | mobile(0.04), lebih(0.03), bni mobile(0.03), banking(0.03), mobile banking(0.02), bank(0.02), transaksi(0.02), kalau(0.02), pakai(0.02), fitur(0.02)
T4     | tunai(0.21), tarik(0.19), tarik tunai(0.17), kartu(0.13), tunai kartu(0.11), fitur(0.08), setor(0.08), fitur tarik(0.07), setor tunai(0.06), atm(0.05)
T5     | mobile(0.13), bni mobile(0.10), banking(0.08), mobile banking(0.07), mending(0.07), lebih

In [21]:
# ============================================================
# Section 6.3 — Representative docs per topic
# ============================================================
# Untuk membantu interpretasi tema, lihat 3 dokumen paling representatif per topik

for topic_id in range(len(topic_info) - 1):
    print(f"\n{'='*90}")
    print(f"TOPIC {topic_id} (n={topic_info.loc[topic_info['Topic'] == topic_id, 'Count'].values[0]} docs)")
    
    # Top words
    top_words = [w for w, _ in topic_model_winner.get_topic(topic_id)[:10]]
    print(f"Top words: {', '.join(top_words)}")
    
    # Representative docs (3 paling representatif)
    rep_docs = topic_model_winner.get_representative_docs(topic_id)[:3]
    print(f"\nRepresentative docs:")
    for i, doc in enumerate(rep_docs, 1):
        # Truncate panjang doc biar gak bikin cell output overwhelming
        doc_preview = doc[:200] + "..." if len(doc) > 200 else doc
        print(f"  {i}. {doc_preview}")


TOPIC 0 (n=450 docs)
Top words: wajah, verifikasi wajah, verifikasi, gagal, gagal terus, wajah gagal, wajah susah, susah banget, padahal, mengikuti

Representative docs:
  1. untuk verifikasi wajah susah banget
  2. susah banget verifikasi wajah gagal terus
  3. susah verifikasi wajah gagal terus

TOPIC 1 (n=416 docs)
Top words: saldo, gagal, gagal saldo, transaksi, transaksi gagal, berkurang, terpotong, pulsa, kepotong, transfer

Representative docs:
  1. transaksi gagal tapi saldo berkurang saat pembelian token listrik
  2. beli pulsa lewat aplikasi saldo berkurang transaksi gagal saldo tidak kembali bagaimana ini
  3. transaksi gagal tapi saldo terpotong

TOPIC 2 (n=399 docs)
Top words: gangguan, sering, gangguan terus, perbaikan, sering error, banget, maintenance, sering gangguan, jelek, seharian

Representative docs:
  1. aplikasi sering gangguan tidak jelas
  2. jelek banget aplikasi sering gangguan dan lemot
  3. sering gangguan atau bisa juga sering perbaikan sistem

TOPIC 3 (